In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model(source = "Video.mp4", show = True, conf = 0.4, save = True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/962) /content/Video.mp4: 384x640 1 person, 337.7ms
vid

In [59]:
""" BYTE TRACK EVALUATION """
import numpy as np
from scipy.optimize import linear_sum_assignment

IOU_THRESH = 0.5


def load_file(filename):
    data = {}

    with open(filename, "r") as f:
        for line in f:
            vals = line.strip().split(",")

            frame = int(vals[0])
            tid = int(vals[1])

            x = float(vals[2])
            y = float(vals[3])
            w = float(vals[4])
            h = float(vals[5])

            data.setdefault(frame, []).append({
                "id": tid,
                "bbox": [x, y, w, h]
            })

    return data


def iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    xa = max(x1, x2)
    ya = max(y1, y2)
    xb = min(x1 + w1, x2 + w2)
    yb = min(y1 + h1, y2 + h2)

    inter = max(0, xb - xa) * max(0, yb - ya)

    union = w1 * h1 + w2 * h2 - inter

    if union <= 0:
        return 0

    return inter / union


def evaluate(gt_file, pred_file):

    gt = load_file(gt_file)
    pred = load_file(pred_file)

    frames = sorted(set(gt.keys()) | set(pred.keys()))

    TP = FP = FN = 0
    IDTP = IDFP = IDFN = 0
    IDSW = 0

    previous_matches = {}

    for frame in frames:

        gts = gt.get(frame, [])
        prs = pred.get(frame, [])

        if len(gts) == 0:
            FP += len(prs)
            IDFP += len(prs)
            continue

        if len(prs) == 0:
            FN += len(gts)
            IDFN += len(gts)
            continue

        cost = np.ones((len(gts), len(prs)))

        for i, g in enumerate(gts):
            for j, p in enumerate(prs):
                cost[i, j] = 1 - iou(g["bbox"], p["bbox"])

        rows, cols = linear_sum_assignment(cost)

        matched_gt = set()
        matched_pr = set()

        current_matches = {}

        for r, c in zip(rows, cols):

            if 1 - cost[r, c] < IOU_THRESH:
                continue

            matched_gt.add(r)
            matched_pr.add(c)

            TP += 1
            IDTP += 1

            gtid = gts[r]["id"]
            prid = prs[c]["id"]

            current_matches[gtid] = prid

            if gtid in previous_matches:
                if previous_matches[gtid] != prid:
                    IDSW += 1

        FN += len(gts) - len(matched_gt)
        FP += len(prs) - len(matched_pr)

        IDFN += len(gts) - len(matched_gt)
        IDFP += len(prs) - len(matched_pr)

        previous_matches = current_matches

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0

    idp = IDTP / (IDTP + IDFP) if IDTP + IDFP else 0
    idr = IDTP / (IDTP + IDFN) if IDTP + IDFN else 0

    idf1 = 2 * idp * idr / (idp + idr) if idp + idr else 0

    mota = 1 - (FP + FN + IDSW) / max(1, sum(len(v) for v in gt.values()))

    print("=" * 55)
    print("Tracking Evaluation")
    print("=" * 55)
    print(f"True Positives : {TP}")
    print(f"False Positives: {FP}")
    print(f"False Negatives: {FN}")
    print(f"Identity Switches: {IDSW}")
    print()
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"MOTA      : {mota:.4f}")
    print(f"IDP       : {idp:.4f}")
    print(f"IDR       : {idr:.4f}")
    print(f"IDF1      : {idf1:.4f}")


if __name__ == "__main__":

    evaluate(
        "ref/gt.txt",
        "ref_bytetrack_submission.txt"
    )

Tracking Evaluation
True Positives : 8107
False Positives: 786
False Negatives: 21086
Identity Switches: 43

Precision : 0.9116
Recall    : 0.2777
MOTA      : 0.2493
IDP       : 0.9116
IDR       : 0.2777
IDF1      : 0.4257


In [62]:
""" BYTE TRACK"""
import cv2
import os
import glob
import yaml
import numpy as np
from ultralytics import YOLO


class ByteTrackGenerator:

    def __init__(
        self,
        image_dir="Images",
        output_file="ref_bytetrack_submission.txt",
        model_name="yolov8x.pt"
    ):

        self.image_dir = image_dir
        self.output_file = output_file
        self.model_name = model_name

        self.config_file = "custom_bytetrack.yaml"

        self.model = None

        self._create_tracker_config()

    ############################################################
    # ByteTrack Configuration
    ############################################################

    def _create_tracker_config(self):

        tracker_cfg = {

            "tracker_type": "bytetrack",

            # Detection thresholds
            "track_high_thresh": 0.35,
            "track_low_thresh": 0.10,
            "new_track_thresh": 0.35,

            # Association
            "match_thresh": 0.75,

            # Keep tracks alive longer
            "track_buffer": 90,

            # Fuse detector confidence with IoU
            "fuse_score": True
        }

        with open(self.config_file, "w") as f:
            yaml.dump(
                tracker_cfg,
                f,
                sort_keys=False
            )

    ############################################################
    # Load YOLO
    ############################################################

    def load_model(self):

        print("=" * 60)
        print("Loading YOLO Detector...")
        print("=" * 60)

        self.model = YOLO(self.model_name)

        print(f"Loaded {self.model_name}")

    ############################################################
    # Read Frames
    ############################################################

    def get_image_list(self):

        image_paths = sorted(
            glob.glob(
                os.path.join(
                    self.image_dir,
                    "*.jpg"
                )
            )
        )

        if len(image_paths) == 0:
            raise RuntimeError(
                f"No jpg images found inside '{self.image_dir}'"
            )

        print(f"Found {len(image_paths)} frames.")

        return image_paths

    ############################################################
    # Detection Filters
    ############################################################

    @staticmethod
    def valid_detection(box, conf):

        x1, y1, x2, y2 = box

        w = x2 - x1
        h = y2 - y1

        # Remove tiny detections
        if w < 15:
            return False

        if h < 30:
            return False

        # Remove weak detections
        if conf < 0.10:
            return False

        return True

    ############################################################
    # xyxy -> MOT format
    ############################################################

    @staticmethod
    def convert_box(box):

        x1, y1, x2, y2 = box

        return (
            x1,
            y1,
            x2 - x1,
            y2 - y1
        )

    ############################################################
    # Write one MOT detection
    ############################################################

    @staticmethod
    def write_detection(
        file,
        frame,
        track_id,
        box
    ):

        x, y, w, h = ByteTrackGenerator.convert_box(box)

        file.write(
            f"{frame},"
            f"{track_id},"
            f"{x:.2f},"
            f"{y:.2f},"
            f"{w:.2f},"
            f"{h:.2f},"
            "-1,-1,-1,-1\n"
        )

    ############################################################
    # Cleanup
    ############################################################

    def cleanup(self):

        if os.path.exists(self.config_file):
            os.remove(self.config_file)
                ############################################################
    # Tracking
    ############################################################

    def generate_submission(self):

        self.load_model()

        image_paths = self.get_image_list()

        print("=" * 60)
        print("Running ByteTrack...")
        print("=" * 60)

        with open(self.output_file, "w") as out_file:

            for frame_idx, img_path in enumerate(image_paths, start=1):

                frame = cv2.imread(img_path)

                if frame is None:
                    continue

                results = self.model.track(
                    source=frame,
                    persist=True,
                    tracker=self.config_file,
                    classes=[0],
                    conf=0.10,
                    iou=0.70,
                    imgsz=1280,
                    augment=True,
                    verbose=False
                )

                result = results[0]

                if (
                    result.boxes is None
                    or result.boxes.id is None
                    or len(result.boxes) == 0
                ):
                    continue

                boxes = result.boxes.xyxy.cpu().numpy()
                ids = result.boxes.id.int().cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()

                # Highest confidence first
                order = np.argsort(-confs)

                boxes = boxes[order]
                ids = ids[order]
                confs = confs[order]

                used_ids = set()

                for box, track_id, conf in zip(
                    boxes,
                    ids,
                    confs
                ):

                    if track_id <= 0:
                        continue

                    if track_id in used_ids:
                        continue

                    if not self.valid_detection(box, conf):
                        continue

                    used_ids.add(track_id)

                    self.write_detection(
                        out_file,
                        frame_idx,
                        int(track_id),
                        box
                    )

                if frame_idx % 25 == 0 or frame_idx == len(image_paths):
                    print(
                        f"Processed "
                        f"{frame_idx}/{len(image_paths)} frames"
                    )

        self.cleanup()

        print("\n" + "=" * 60)
        print("Tracking Complete")
        print("=" * 60)
        print(f"Results saved to {self.output_file}")


############################################################
# Main
############################################################

if __name__ == "__main__":

    tracker = ByteTrackGenerator(
        image_dir="Images4",
        output_file="ref_bytetrack_submission.txt",
        model_name="yolov8x.pt"
    )

    tracker.generate_submission()

Loading YOLO Detector...
Loaded yolov8x.pt
Found 1050 frames.
Running ByteTrack...
Processed 25/1050 frames
Processed 50/1050 frames
Processed 75/1050 frames
Processed 100/1050 frames
Processed 125/1050 frames
Processed 150/1050 frames
Processed 175/1050 frames
Processed 200/1050 frames
Processed 225/1050 frames
Processed 250/1050 frames
Processed 275/1050 frames
Processed 300/1050 frames
Processed 325/1050 frames
Processed 350/1050 frames
Processed 375/1050 frames
Processed 400/1050 frames
Processed 425/1050 frames
Processed 450/1050 frames
Processed 475/1050 frames
Processed 500/1050 frames
Processed 525/1050 frames
Processed 550/1050 frames
Processed 575/1050 frames
Processed 600/1050 frames
Processed 625/1050 frames
Processed 650/1050 frames
Processed 675/1050 frames
Processed 700/1050 frames
Processed 725/1050 frames
Processed 750/1050 frames
Processed 775/1050 frames
Processed 800/1050 frames
Processed 825/1050 frames
Processed 850/1050 frames
Processed 875/1050 frames
Processed 